<a href="https://colab.research.google.com/github/Yashwant-Vadhan/Indian-Cattle-Breeds_SIH25004/blob/main/Dataset_Cattle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install icrawler pillow opencv-python imagehash tqdm transformers torch torchvision


In [ ]:
base_path = "/content/drive/MyDrive/Cattle_Dataset"

In [ ]:
# Target Indian cattle breeds
cattle_breeds = [
    "Alambadi cattle",
    "Amrit Mahal cattle",
    "Bachaur cattle",
    "Bargur cattle",
    "Dangi cattle",
    "Deoni cattle",
    "Gaolao cattle",
    "Gidda cattle",
    "Gir cattle",
    "Hallikar cattle",
    "Hariana cattle",
    "Kangayam cattle",
    "Kankrej cattle",
    "Kasaragod Dwarf",
    "Kenkatha cattle",
    "Kherigarh cattle",
    "Khillari cattle",
    "Krishna Valley cattle",
    "Malvi cattle",
    "Mewati cattle",
    "Nagori cattle",
    "Nimari cattle",
    "Ongole cattle",
    "Ponwar cattle",
    "Pulikulam cattle",
    "Rathi cattle",
    "Red Kandhari cattle",
    "Red Sindhi cattle",
    "Sahiwal cattle",
    "Siri cattle",
    "Tharparkar cattle",
    "Vechur cattle",
    "Motu cattle",
    "Ghumusari cattle",
    "Binjharpuri cattle",
    "Khariar cattle",
    "Kosali cattle",
    "Belahi cattle",
    "Gangatiri cattle",
    "Badri cattle",
    "Lakhimi cattle",
    "Ladakhi cattle",
    "Konkan Kapila cattle",
    "Poda Thurpu cattle",
    "Nari cattle",
    "Dagri cattle",
    "Thutho cattle",
    "Shweta Kapila cattle",
    "Himachali Pahari cattle",
    "Purnea cattle",
    "Umblachery cattle"
]


In [ ]:
import os
os.makedirs(base_path, exist_ok=True)

# 1) Install dependencies
!pip install -q icrawler pillow requests certifi tqdm

# 2) Patch SSL environment (helpful fallback)
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

# 3) Define custom downloader for icrawler that uses requests.Session(verify=False)
from icrawler import downloader
import requests
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)  # silence warnings

class MyDownloader(downloader.Downloader):
    """
    Custom icrawler downloader that uses a requests.Session with verification turned off.
    This helps to download images from websites with expired/invalid SSL certificates.
    """
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Create a session that does not verify SSL certs
        self.session = requests.Session()
        self.session.verify = False
        # polite default headers (helps with some sites)
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
        })

    def get_response(self, task, timeout=10, **kwargs):
        """
        task is a dict with at least 'file_url' key (icrawler internals).
        We override to fetch via requests.Session with verify=False.
        """
        url = task.get('file_url') if isinstance(task, dict) else task
        try:
            # stream=True to avoid loading whole file in memory; icrawler handles saving
            return self.session.get(url, stream=True, timeout=timeout)
        except Exception as e:
            # Let icrawler handle the exception flow (it retries based on its config)
            raise

# 4) Import crawlers and define download function
from icrawler.builtin import GoogleImageCrawler, BingImageCrawler
from time import sleep

def download_images_for_breed(breed, max_per_keyword=500, pause=1.0):
    """
    Download images for a breed using multiple keywords and both Google+Bing crawlers.
    max_per_keyword: number to request from each search engine per keyword (icrawler caps may apply)
    pause: seconds to wait between engine requests to be polite
    """
    save_dir = os.path.join(base_path, breed.replace(' ', '_'))
    os.makedirs(save_dir, exist_ok=True)

    # generate a small set of related keywords to increase coverage
    keywords = [
        breed,
        breed.replace("cattle", "cow"),
        breed.replace("cattle", "bull"),
        breed + " India",
        breed + " breed"
    ]
    # deduplicate keywords
    keywords = list(dict.fromkeys(keywords))

    for kw in keywords:
        print(f"\n>> Crawling keyword: {kw}  -> saving to {save_dir}")

        # Google crawler with custom downloader
        g = GoogleImageCrawler(storage={"root_dir": save_dir}, downloader_cls=MyDownloader)
        try:
            g.crawl(keyword=kw, max_num=max_per_keyword, file_idx_offset=0)
        except Exception as e:
            print("Google crawl error (continuing):", str(e))

        # small polite pause
        sleep(pause)

        # Bing crawler with custom downloader
        b = BingImageCrawler(storage={"root_dir": save_dir}, downloader_cls=MyDownloader)
        try:
            b.crawl(keyword=kw, max_num=max_per_keyword, file_idx_offset=0)
        except Exception as e:
            print("Bing crawl error (continuing):", str(e))

        sleep(pause)
# download_images_for_breed("Umblachery cattle", max_per_keyword=500)
for breed in cattle_breeds:
    download_images_for_breed(breed, max_per_keyword=500)  # adjust numbers as needed
    print(f"Completed downloads for {breed}")


ERROR:downloader:Response status code 404, file https://www.apnikheti.com/en/pn/livestock/cow/umblachery","15.jpg
ERROR:downloader:Response status code 403, file https://www.researchgate.net/profile/Saravanan-Mani-2/publication/386068158/figure/tbl2/AS:11431281293287821@1732812907481/Blood-gas-analysis-of-Umblachery-bull-with-uroabdomen_Q320.jpg
Exception in thread parser-001:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/icrawler/parser.py", line 93, in worker_exec
    for task in self.parse(response, **kwargs):
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not iterable
ERROR:downloader:Response status code 400, file https://media.istockphoto.com/id/185221235/vector/19th-century-engraving-of-a-zebu-or-brahmin-bull.jpg
ERROR


>> Crawling keyword: Umblachery cattle  -> saving to /content/drive/MyDrive/Cattle_Dataset/Umblachery_cattle


ERROR:downloader:Response status code 404, file https://www.apnikheti.com/en/pn/livestock/cow/umblachery","8298idea99umblacherry.jpg
ERROR:downloader:Response status code 404, file https://a0.anyrgb.com/png
ERROR:downloader:Response status code 400, file https://media.istockphoto.com/id/185221235/vector/19th-century-engraving-of-a-zebu-or-brahmin-bull.jpg
ERROR:downloader:Response status code 404, file https://www.apnikheti.com/en/pn/livestock/cow/umblachery","15.jpg
Exception in thread parser-001:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/icrawler/parser.py", line 93, in worker_exec
    for task in self.parse(response, **kwargs):
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not iterable
ERROR:downloader:Response status


>> Crawling keyword: Umblachery cow  -> saving to /content/drive/MyDrive/Cattle_Dataset/Umblachery_cattle


ERROR:downloader:Response status code 404, file https://a0.anyrgb.com/png
Exception in thread parser-001:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/icrawler/parser.py", line 93, in worker_exec
    for task in self.parse(response, **kwargs):
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not iterable


...

ERROR:downloader:Response status code 404, file https://a0.anyrgb.com/png
Exception in thread parser-001:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/icrawler/parser.py", line 93, in worker_exec
    for task in self.parse(response, **kwargs):
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not iterable



>> Crawling keyword: Umblachery cattle India  -> saving to /content/drive/MyDrive/Cattle_Dataset/Umblachery_cattle


ERROR:downloader:Response status code 403, file https://i1.rgstatic.net/publication/338595939_Influence_of_Ageing_and_Regional_Differences_on_Draught_Performance_of_Umblachery_Cattle/links/5e74baf6299bf1b4197d5b55/largepreview.png
Exception in thread parser-001:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/icrawler/parser.py", line 93, in worker_exec
    for task in self.parse(response, **kwargs):
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not iterable
ERROR:downloader:Response status code 403, file https://i1.rgstatic.net/publication/263216049_Chromosome_profile_of_Umblachery_cattle/links/5649652608aef646e6d23567/largepreview.png
ERROR:downloader:Exception caught when downloading file https://www.usnews.com/dims4/USNEWS


>> Crawling keyword: Umblachery cattle breed  -> saving to /content/drive/MyDrive/Cattle_Dataset/Umblachery_cattle


ERROR:downloader:Response status code 403, file https://i1.rgstatic.net/publication/338595939_Influence_of_Ageing_and_Regional_Differences_on_Draught_Performance_of_Umblachery_Cattle/links/5e74baf6299bf1b4197d5b55/largepreview.png
Exception in thread parser-001:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/icrawler/parser.py", line 93, in worker_exec
    for task in self.parse(response, **kwargs):
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: 'NoneType' object is not iterable


In [ ]:
import os
cnt = 0
for breed in cattle_breeds:
    path = f"{base_path}/{breed.replace(' ', '_')}"
    count = len(os.listdir(path))
    cnt += count
    print(f"{breed}: {count} images")
print("Total imgs: ", cnt)

Alambadi cattle: 231 images
Amrit Mahal cattle: 179 images
Bachaur cattle: 258 images
Bargur cattle: 230 images
Dangi cattle: 213 images
Deoni cattle: 202 images
Gaolao cattle: 289 images
Gidda cattle: 264 images
Gir cattle: 227 images
Hallikar cattle: 224 images
Hariana cattle: 223 images
Kangayam cattle: 258 images
Kankrej cattle: 227 images
Kasaragod Dwarf: 222 images
Kenkatha cattle: 233 images
Kherigarh cattle: 304 images
Khillari cattle: 243 images
Krishna Valley cattle: 256 images
Malvi cattle: 301 images
Mewati cattle: 278 images
Nagori cattle: 277 images
Nimari cattle: 291 images
Ongole cattle: 254 images
Ponwar cattle: 279 images
Pulikulam cattle: 274 images
Rathi cattle: 249 images
Red Kandhari cattle: 248 images
Red Sindhi cattle: 230 images
Sahiwal cattle: 238 images
Siri cattle: 225 images
Tharparkar cattle: 226 images
Vechur cattle: 219 images
Motu cattle: 178 images
Ghumusari cattle: 310 images
Binjharpuri cattle: 312 images
Khariar cattle: 268 images
Kosali cattle: 249

In [ ]:
import os
from tqdm import tqdm
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel

# Load CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def is_correct_breed(image_path, breed_name, threshold=0.25):
    try:
        image = Image.open(image_path).convert("RGB")

        # Define positive prompts for the target breed
        positive_prompts = [
            f"a photo of a {breed_name} cow",
            f"a photo of a {breed_name} bull",
            f"a photo of {breed_name} cattle",
        ]

        # Generic cattle prompts (backup positives)
        generic_prompts = [
            "a photo of a cow",
            "a photo of a bull",
            "a photo of cattle",
        ]

        # Negative prompts to filter junk
        negative_prompts = [
            "captions", "quotes",
            "a photo of a person",
            "a photo of humans",
            "a photo of food",
            "a photo of a box",
            "a photo of a building",
            "a photo of scenery",
            "a photo of a goat",
            "a photo of a buffalo",
            "a photo of a horse",
            "not a cow",
        ]

        all_prompts = positive_prompts + generic_prompts + negative_prompts

        inputs = processor(
            text=all_prompts,
            images=image,
            return_tensors="pt",
            padding=True
        ).to(device)

        outputs = model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1).detach().cpu().numpy()[0]

        # Positive = breed-specific + generic cattle
        breed_score = sum(probs[:len(positive_prompts)])   # breed-specific
        cattle_score = sum(probs[len(positive_prompts):len(positive_prompts)+len(generic_prompts)]) # generic cattle

        final_score = breed_score + 0.5 * cattle_score  # prioritize breed score, allow generic fallback
        return final_score > threshold

    except Exception as e:
        return False

def filter_wrong_breeds(folder_path, breed_name):
    removed = 0
    for file in tqdm(os.listdir(folder_path)):
        file_path = os.path.join(folder_path, file)
        if not is_correct_breed(file_path, breed_name):
            os.remove(file_path)
            removed += 1
    print(f"Removed {removed} irrelevant/wrong-breed images from {folder_path}")

# Apply filter breed-wise
breed = "Umblachery cattle"
folder = f"{base_path}/{breed.replace(' ', '_')}"
filter_wrong_breeds(folder, breed)
# for breed in cattle_breeds:
#     folder = f"{base_path}/{breed.replace(' ', '_')}"
#     filter_wrong_breeds(folder, breed)

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]


 82%|████████▏ | 239/291 [02:39<00:32,  1.62it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(

100%|██████████| 291/291 [03:13<00:00,  1.50it/s]

Removed 14 irrelevant/wrong-breed images from /content/drive/MyDrive/Cattle_Dataset/Umblachery_cattle
